In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Validação dos lotes de embeddings gerados pelo Sentence Transformers

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm


PASTA_EMBEDDINGS = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/embeddings"
)

DIM_ESPERADA = 384

arquivos = sorted(PASTA_EMBEDDINGS.glob("*.parquet"))

print("Total de lotes encontrados:", len(arquivos))


problemas = []
total_embeddings = 0


for arquivo in tqdm(arquivos, desc="Validando lotes"):

    try:
        df = pd.read_parquet(arquivo)

        if "embedding" not in df.columns:
            problemas.append(
                (arquivo.name, "coluna embedding ausente")
            )
            continue


        embeddings = np.vstack(df["embedding"].values)


        total_embeddings += len(embeddings)


        if embeddings.shape[1] != DIM_ESPERADA:
            problemas.append(
                (
                    arquivo.name,
                    f"dimensão encontrada {embeddings.shape}"
                )
            )


        if embeddings.dtype != np.float32:
            problemas.append(
                (
                    arquivo.name,
                    f"dtype encontrado {embeddings.dtype}"
                )
            )


        if not np.isfinite(embeddings).all():
            problemas.append(
                (
                    arquivo.name,
                    "possui NaN ou infinito"
                )
            )


    except Exception as e:
        problemas.append(
            (arquivo.name, str(e))
        )


print("\n========================")
print("Total embeddings verificados:", total_embeddings)
print("Problemas encontrados:", len(problemas))


if problemas:
    print("\nProblemas encontrados:")
    for p in problemas[:10]:
        print(p)

else:
    print("Todos os embeddings estão válidos!")

Verificação de normalização

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm


PASTA_EMBEDDINGS = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/embeddings"
)


arquivos = sorted(PASTA_EMBEDDINGS.glob("*.parquet"))


normas = []

print("Total de lotes:", len(arquivos))


for arquivo in tqdm(arquivos, desc="Verificando normalização"):

    df = pd.read_parquet(arquivo)

    embeddings = np.vstack(
        df["embedding"].values
    ).astype(np.float32)


    lote_normas = np.linalg.norm(
        embeddings,
        axis=1
    )


    normas.extend(lote_normas)


normas = np.array(normas)


print("\n========================")
print("Total embeddings analisados:", len(normas))

print("\nNorma mínima:", normas.min())
print("Norma máxima:", normas.max())
print("Norma média:", normas.mean())

print("\nDesvio padrão:", normas.std())

Verificando se a quantidade de registros no SQLite e embeddings são iguais (768.348)

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from tqdm.auto import tqdm

BANCO_SQLITE = Path(
    "/content/drive/MyDrive/TCC_Chatbot/SQLite/banco_respostas.sqlite"
)

PASTA_EMBEDDINGS = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/embeddings"
)


conn = sqlite3.connect(BANCO_SQLITE)

cursor = conn.cursor()

cursor.execute(
    "SELECT COUNT(*) FROM respostas"
)

total_sqlite = cursor.fetchone()[0]

conn.close()

arquivos = sorted(
    PASTA_EMBEDDINGS.glob("*.parquet")
)


total_embeddings = 0


for arquivo in tqdm(
    arquivos,
    desc="Contando embeddings"
):

    df = pd.read_parquet(
        arquivo,
        columns=["embedding"]
    )

    total_embeddings += len(df)

print("\n========================")
print("Registros no SQLite:", total_sqlite)
print("Embeddings gerados:", total_embeddings)


if total_sqlite == total_embeddings:
    print("\nQuantidades correspondem!")
    print("SQLite e FAISS poderão ficar alinhados.")

else:
    print("\nQuantidades diferentes!")
    print(
        "Diferença:",
        abs(total_sqlite - total_embeddings)
    )

Teste com 100 registros aleatórios

In [ ]:
import sqlite3
import pandas as pd
import random


BANCO_SQLITE = "/content/drive/MyDrive/TCC_Chatbot/SQLite/banco_respostas.sqlite"

conn = sqlite3.connect(BANCO_SQLITE)

df_sqlite = pd.read_sql_query(
    """
    SELECT id, question
    FROM respostas
    ORDER BY id
    """,
    conn
)

conn.close()

import glob

arquivos = sorted(
    glob.glob(
        "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/embeddings/*.parquet"
    )
)

df_emb = pd.concat(
    [
        pd.read_parquet(
            arquivo,
            columns=["question"]
        )
        for arquivo in arquivos
    ],
    ignore_index=True
)


indices = random.sample(
    range(len(df_sqlite)),
    100
)


erros = []

for i in indices:

    q_sqlite = df_sqlite.iloc[i]["question"]
    q_emb = df_emb.iloc[i]["question"]

    if q_sqlite != q_emb:
        erros.append(i)


print("Registros testados:", len(indices))
print("Diferenças encontradas:", len(erros))


if len(erros) == 0:
    print("Ordem SQLite ↔ Embeddings confirmada!")

else:
    print("Problemas nos índices:", erros[:10])

Criação do índice FAISS

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import faiss
from tqdm.auto import tqdm
import json
import gc


PASTA_EMBEDDINGS = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/embeddings"
)


PASTA_SAIDA = Path(
    "/content/drive/MyDrive/TCC_Chatbot/FAISS"
)

PASTA_SAIDA.mkdir(
    parents=True,
    exist_ok=True
)


arquivos = sorted(
    PASTA_EMBEDDINGS.glob("*.parquet")
)


DIMENSAO = 384


index = faiss.IndexFlatIP(
    DIMENSAO
)


total = 0


print("Criando índice FAISS...")


for arquivo in tqdm(
    arquivos,
    desc="Adicionando embeddings"
):

    df = pd.read_parquet(
        arquivo,
        columns=["embedding"]
    )


    embeddings = np.vstack(
        df["embedding"].values
    ).astype(
        np.float32
    )


    if embeddings.shape[1] != DIMENSAO:
        raise ValueError(
            f"Dimensão inválida em {arquivo.name}: {embeddings.shape}"
        )


    index.add(
        embeddings
    )


    total += embeddings.shape[0]


    del df
    del embeddings

    gc.collect()



print("\n====================")
print("Vetores adicionados:", total)
print("Dimensão:", index.d)


caminho_index = PASTA_SAIDA / "indice_faiss.index"


faiss.write_index(
    index,
    str(caminho_index)
)

metadados = {
    "total_vetores": total,
    "dimensao": DIMENSAO,
    "tipo": "IndexFlatIP",
    "normalizado": True
}


with open(
    PASTA_SAIDA / "metadados.json",
    "w"
) as f:

    json.dump(
        metadados,
        f,
        indent=4
    )


print("\nÍndice FAISS salvo em:")
print(caminho_index)

Verificação do indice_faiss.index

In [ ]:
import faiss

CAMINHO_INDEX = "/content/drive/MyDrive/TCC_Chatbot/FAISS/indice_faiss.index"

index = faiss.read_index(CAMINHO_INDEX)

print("Vetores no índice:", index.ntotal)

Consultando a primeira pergunta do banco

In [ ]:
import pandas as pd
import numpy as np
import sqlite3

conn = sqlite3.connect(
    "/content/drive/MyDrive/TCC_Chatbot/SQLite/banco_respostas.sqlite"
)

pergunta = pd.read_sql_query(
    """
    SELECT question
    FROM respostas
    WHERE id = 0
    """,
    conn
).iloc[0]["question"]

conn.close()


print("Pergunta consultada:")
print(pergunta)

Verificando os vetores com o modelo do Sentence Transformers

In [ ]:
from sentence_transformers import SentenceTransformer


modelo = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)


vetor = modelo.encode(
    pergunta,
    normalize_embeddings=True
)


vetor = np.array(
    [vetor],
    dtype=np.float32
)

Confirmação das similaridades dos índices (SQLite = FAISS)

In [ ]:
distancias, indices = index.search(
    vetor,
    k=5
)


print("Índices encontrados:")
print(indices)

print("\nSimilaridades:")
print(distancias)